# 07. Preprocessing Test

This notebook tests the preprocessing module on the sample pose CSV.

Preprocessing runs after annotation and before normalization in the pipeline:

```
Validation → Annotation → Exercise Definition → Preprocessing → Normalization → Motion Attribution → Features
```

The module identifies low-reliability landmark detections and corrects short-term noise.
It does **not** change movement quality — it only fixes data quality problems.

Output columns added by preprocessing:

- `<landmark>_reliable` — per-landmark per-frame reliability mask (bool)
- `preprocessing_valid` — frame-level summary (bool)
- `preprocessing_note` — reason if invalid (str)
- `swap_corrected` — frame-level label swap flag (bool)

This notebook assumes that the previous checks are already working:

- 00_environment_check
- 01_data_loading_test
- 02_validation_test
- 03_raw_visualization_test
- 04_normalization_test
- 05_annotation_mask_test
- 06_exercise_definition_test

> **Status:** The `preprocessing` module is not yet implemented.
> Running the pipeline with `preprocessing.enabled: true` raises `NotImplementedError`.
> This notebook documents the design and will be updated when the module is ready.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path

import yaml

from movement.annotation import apply_annotation, load_annotation_csv
from movement.config import LANDMARKS, make_coordinate_columns, make_required_columns, make_visibility_columns
from movement.io import load_pose_csv
from movement.pipeline import PipelineConfig, PreprocessingConfig, load_pipeline_config
from movement.validation import run_basic_validation

## Data Setup

Load pose CSV, run validation, and apply annotation so that preprocessing receives
a dataframe with `exercise_type` and `pattern` context columns.

This mirrors the pipeline order: validation → annotation → (preprocessing) → normalization.

In [ ]:
csv_path = "../data/sample/mediapipe_forward_bend_sample.csv"
ann_path = "../data/sample/mediapipe_forward_bend_sample_annotation.csv"

df = load_pose_csv(csv_path)
print(f"loaded: {df.shape[0]} frames, {df.shape[1]} columns")

In [ ]:
val_report = run_basic_validation(
    df=df,
    required_columns=make_required_columns(LANDMARKS),
    coordinate_columns=make_coordinate_columns(LANDMARKS),
    visibility_columns=make_visibility_columns(LANDMARKS),
)
print("validation passed:", val_report["passed"])

In [ ]:
ann_df = load_annotation_csv(ann_path)
annotated_df, ann_report = apply_annotation(df, ann_df)

print(f"annotated dataframe: {annotated_df.shape[0]} frames, {annotated_df.shape[1]} columns")
print("exercise_type:", annotated_df["exercise_type"].dropna().unique().tolist())
print("pattern:      ", annotated_df["pattern"].dropna().unique().tolist())

## Current Status: Module Not Implemented

The pipeline raises `NotImplementedError` when `preprocessing.enabled` is `True`.

This is the expected behaviour during development.
The preprocessing step must be implemented before this test can verify actual output.

In [ ]:
from movement.pipeline import run_pipeline

config_not_implemented = PipelineConfig()
config_not_implemented.preprocessing = PreprocessingConfig(enabled=True)

try:
    run_pipeline(annotated_df, config=config_not_implemented)
except NotImplementedError as e:
    print("NotImplementedError raised as expected:")
    print(" ", e)

## Config Preview

Preprocessing is configured via `configs/pipeline_default.yaml`.
The current default has `preprocessing.enabled: false`.

Once implemented, the config controls:

- reliability detection thresholds (visibility, segment length, velocity)
- frame-level left-right swap detection (exercise-aware — only for alternating exercises)
- short-gap interpolation over reliability-masked frames
- optional smoothing (rolling median or moving average)

Kalman filtering is planned but disabled until simpler methods are characterised.

In [ ]:
config_path = Path("../configs/pipeline_default.yaml")

with open(config_path, encoding="utf-8") as f:
    raw_config = yaml.safe_load(f)

preprocessing_section = raw_config.get("preprocessing", {})
print(json.dumps(preprocessing_section, indent=2))

## Expected Output Columns

When the module is implemented, preprocessing will add the following columns to the dataframe.

**Per-landmark reliability mask:**

```text
<landmark>_reliable     bool    True if landmark is reliable at this frame
```

For example: `left_knee_reliable`, `right_wrist_reliable`.

**Frame-level summary:**

```text
preprocessing_valid     bool    True if the frame is usable after preprocessing
preprocessing_note      str     reason string if invalid (e.g. 'low_visibility:left_knee')
swap_corrected          bool    True if left-right labels were swapped at this frame
```

## Exercise-Aware Branching

Preprocessing reads the `pattern` column from annotation to decide whether to
enable frame-level left-right swap detection.

```text
pattern = bilateral      swap detection is skipped
pattern = alternating    swap detection is enabled
pattern unknown          falls back to bilateral (safe default)
```

For the sample data (`forward_bend`, `bilateral`), swap detection will be skipped.

## Interpretation

Once the module is implemented, expected checks are:

- `preprocessing_valid` is `True` for all frames in the clean sample data
- `swap_corrected` is `False` for all frames (bilateral exercise, no swap detection)
- `<landmark>_reliable` matches expectations for high-visibility frames
- preprocessing report records: method, exercise_type, pattern, reliability summary, interpolation summary
- `preprocessing_note` is empty or `None` when no issues are found

The original `frame` and `timestamp` columns must remain unchanged after preprocessing.